# Starbucks Customer Sentiment Analysis

This notebook processes and analyzes customer reviews of Starbucks using sentiment analysis techniques and visualizations.

## Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('/content/Starbucks_reviews_data.csv')
df.head()

## 🔍 Visualize Missing Data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values Heatmap')
plt.show()

## Data Cleaning

In [ ]:
# Drop irrelevant columns if present
if 'Image_Links' in df.columns:
    df = df.drop(columns=['Image_Links'])

# Convert date
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'].str.replace('Reviewed ', '', regex=False), errors='coerce')

# Drop missing ratings
df = df.dropna(subset=['Rating'])
df.head()

## Text Preprocessing

In [ ]:
import re

df['Cleaned_Review'] = df['Review'].astype(str).str.lower()
df['Cleaned_Review'] = df['Cleaned_Review'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))
df[['Review', 'Cleaned_Review']].head()

## Sentiment Analysis with TextBlob

In [ ]:
from textblob import TextBlob

df['Polarity'] = df['Cleaned_Review'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['Sentiment'] = df['Polarity'].apply(lambda x: 'positive' if x > 0.1 else 'negative' if x < -0.1 else 'neutral')
df[['Cleaned_Review', 'Polarity', 'Sentiment']].head()

## Sentiment Distribution

In [ ]:
sns.countplot(data=df, x='Sentiment', order=['positive', 'neutral', 'negative'])
plt.title('Sentiment Distribution')
plt.show()

## Sentiment Over Time

In [ ]:
df['Month'] = df['Date'].dt.to_period('M')
df.groupby(['Month', 'Sentiment']).size().unstack().plot(kind='line', figsize=(10, 5))
plt.title('Monthly Sentiment Trend')
plt.ylabel('Review Count')
plt.show()

## Word Cloud for Each Sentiment

In [ ]:
from wordcloud import WordCloud

for sentiment in ['positive', 'neutral', 'negative']:
    text = ' '.join(df[df['Sentiment'] == sentiment]['Cleaned_Review'].dropna())
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud for {sentiment.capitalize()} Reviews")
    plt.show()

## Save Final Processed Data

In [ ]:
df.to_csv('/content/sentiment_reviews.csv', index=False)

## 🔠 Top Keywords by Sentiment

In [ ]:
from collections import Counter
from nltk.corpus import stopwords
from nltk import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
def get_top_n_words(corpus, n=None):
    words = [word for text in corpus for word in word_tokenize(text) if word.isalpha() and word not in stop_words]
    return Counter(words).most_common(n)

for sentiment in ['positive', 'neutral', 'negative']:
    words = df[df['Sentiment'] == sentiment]['Cleaned_Review'].dropna().tolist()
    top_words = get_top_n_words(words, 10)
    print(f"Top words for {sentiment} sentiment:")
    print(top_words)


## ✅ Summary & Next Steps
- We performed text cleaning, sentiment analysis, and visualized trends over time.
- Word clouds and keyword frequency gave us insight into customer feedback.
- The processed dataset can now be used to build dashboards or feed into ML models.